# FaceGallery: Reference-Based Group Photo Retrieval

This notebook demonstrates a face retrieval pipeline: use one reference face image to find photographs in a gallery that contain a matching identity. It uses InsightFace (`buffalo_l`) for face detection and 512-dimensional face embeddings, then cosine similarity for matching.

**Dataset note:** The experiment uses the `lfw-deepfunneled` subset from the LFW Kaggle dataset. The notebook creates synthetic multi-face composites from individual LFW portraits; these are not naturally captured group/event photographs.

**Execution note:** The saved implementation explicitly selects `CPUExecutionProvider`. The notebook was run in Kaggle and expects the dataset under `/kaggle/input/datasets/jessicali9530/lfw-dataset/`.

**Evaluation note:** The threshold sweep uses the same 30 sampled queries for all thresholds. Treat the reported scores as preliminary, not as an independent held-out benchmark.


In [ ]:
%pip install -q insightface onnxruntime opencv-python-headless scikit-learn numpy matplotlib


In [ ]:
# ============================================================
# Reference-Based Group Photo Retrieval — FaceGallery
# Developed and tested in a Kaggle notebook environment
# ============================================================

# --- Step 0: Install dependencies (run once in a notebook cell) ---
# !pip install insightface faiss-cpu onnxruntime-gpu scikit-learn opencv-python-headless -q

import os
import cv2
import numpy as np
from sklearn.datasets import fetch_lfw_people
import insightface
from insightface.app import FaceAnalysis

In [ ]:
import os
import cv2
import numpy as np

print("Loading LFW dataset from Kaggle input...")

LFW_DIR = "/kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"

image_bank = []
labels = []
label_names = []
name_to_id = {}

for person_name in sorted(os.listdir(LFW_DIR)):
    person_dir = os.path.join(LFW_DIR, person_name)
    if not os.path.isdir(person_dir):
        continue
    img_files = os.listdir(person_dir)
    if len(img_files) < 20:        # same min_faces_per_person=20 filter as before
        continue

    if person_name not in name_to_id:
        name_to_id[person_name] = len(label_names)
        label_names.append(person_name)
    person_id = name_to_id[person_name]

    for img_file in img_files:
        img_path = os.path.join(person_dir, img_file)
        img = cv2.imread(img_path)  # already BGR uint8, no conversion needed
        if img is None:
            continue
        image_bank.append(img)
        labels.append(person_id)

labels = np.array(labels)
names = np.array(label_names)

print(f"Loaded {len(image_bank)} images across {len(names)} identities")

In [ ]:
# ------------------------------------------------------------
# Step 2: Initialize detector + embedder (InsightFace buffalo_l)
# Single model handles both detection AND embedding
# ------------------------------------------------------------
print("Loading InsightFace model...")
app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])  # swap to CUDAExecutionProvider on Kaggle GPU
app.prepare(ctx_id=0, det_size=(640, 640))


In [ ]:
# ------------------------------------------------------------
# Step 3: Detect + embed every face in every image
# Store: embedding vector, source image index, identity label (for eval only)
# ------------------------------------------------------------
print("Detecting and embedding all faces...")

embedding_bank = []      # list of 512-dim vectors
meta_bank = []           # list of dicts: {img_idx, true_label}

for idx, img in enumerate(image_bank):
    faces = app.get(img)  # returns list of Face objects with .embedding, .bbox, .det_score
    for face in faces:
        if face.det_score < 0.5:      # skip low-confidence detections
            continue
        embedding_bank.append(face.embedding)
        meta_bank.append({"img_idx": idx, "true_label": labels[idx]})

    if idx % 200 == 0:
        print(f"  processed {idx}/{len(image_bank)}")

embedding_bank = np.array(embedding_bank)  # shape: (num_faces, 512)
print(f"Total faces embedded: {len(embedding_bank)}")

# Normalize embeddings for cosine similarity via dot product
embedding_bank_norm = embedding_bank / np.linalg.norm(embedding_bank, axis=1, keepdims=True)


In [ ]:
# ------------------------------------------------------------
# Step 4: Query function — give it one photo, get back matching images
# ------------------------------------------------------------
def query_person(query_img, threshold=0.45, top_k=None):
    """
    query_img: uint8 BGR image containing the person's face
    threshold: cosine similarity cutoff (tune this!)
    top_k: if set, return top K matches instead of threshold-based
    Returns: list of (img_idx, similarity_score)
    """
    faces = app.get(query_img)
    if len(faces) == 0:
        raise ValueError("No face detected in query image")

    query_emb = faces[0].embedding
    query_emb_norm = query_emb / np.linalg.norm(query_emb)

    sims = embedding_bank_norm @ query_emb_norm  # cosine similarity, shape (num_faces,)

    if top_k:
        top_indices = np.argsort(sims)[::-1][:top_k]
        results = [(meta_bank[i]["img_idx"], sims[i]) for i in top_indices]
    else:
        matches = np.where(sims >= threshold)[0]
        results = [(meta_bank[i]["img_idx"], sims[i]) for i in matches]
        results.sort(key=lambda x: -x[1])

    # dedupe by img_idx (a photo could match multiple times if crops overlap)
    seen = {}
    for img_idx, score in results:
        if img_idx not in seen or score > seen[img_idx]:
            seen[img_idx] = score
    return sorted(seen.items(), key=lambda x: -x[1])

In [ ]:
# ------------------------------------------------------------
# Step 5: Evaluation — precision/recall against known identity labels
# THIS IS THE PART THAT MATTERS. Don't skip this like the WebHaluCheck
# threshold mistake — tune threshold properly before trusting results.
# ------------------------------------------------------------
def evaluate_threshold(threshold, n_test_queries=30):
    """
    Pick random images as queries, check if system correctly retrieves
    all other images of the same identity, without pulling in wrong people.
    """
    rng = np.random.default_rng(42)
    test_indices = rng.choice(len(image_bank), size=n_test_queries, replace=False)

    precisions, recalls = [], []

    for qi in test_indices:
        query_label = labels[qi]
        ground_truth = set(i for i in range(len(image_bank)) if labels[i] == query_label and i != qi)

        if len(ground_truth) == 0:
            continue  # skip identities with only 1 image

        results = query_person(image_bank[qi], threshold=threshold)
        predicted = set(img_idx for img_idx, score in results if img_idx != qi)

        tp = len(predicted & ground_truth)
        fp = len(predicted - ground_truth)
        fn = len(ground_truth - predicted)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls)

In [ ]:
# Sweep thresholds to find the sweet spot — same lesson from your
# NLI threshold problem in WebHaluCheck, don't just pick 0.5 and hope
print("\nSweeping thresholds...")
for t in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    p, r = evaluate_threshold(t)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    print(f"threshold={t:.2f}  precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}")


In [ ]:
# ------------------------------------------------------------
# Step 6: Save embeddings + metadata so you don't recompute every run
# ------------------------------------------------------------
np.save("embedding_bank.npy", embedding_bank)
import pickle
with open("meta_bank.pkl", "wb") as f:
    pickle.dump(meta_bank, f)

print("\nDone. Embeddings saved. Pick your best threshold from the sweep above,")
print("then plug it into query_person() for the actual demo.")

In [ ]:
# ============================================================
# Step 7: Multi-face / group-photo testing (synthetic composites from LFW)
# Run this AFTER Steps 1-6 have completed successfully
# ============================================================

import random


In [ ]:
# ------------------------------------------------------------
# 7a: Build synthetic "group photos" by placing multiple LFW
# face images side-by-side / in a grid on one canvas.
# This tests: does the detector still find EVERY face when
# there's more than one in frame, at different positions/sizes.
# ------------------------------------------------------------

def make_composite(image_bank, labels, n_people=4, cell_size=200, rng=None):
    """
    Randomly pick n_people images and arrange them in a grid on one canvas.
    Returns: composite image, list of (person_label, position) for verification
    """
    if rng is None:
        rng = random.Random()

    indices = rng.sample(range(len(image_bank)), n_people)

    cols = int(np.ceil(np.sqrt(n_people)))
    rows = int(np.ceil(n_people / cols))

    canvas = np.full((rows * cell_size, cols * cell_size, 3), 40, dtype=np.uint8)  # dark gray bg

    placed_labels = []
    for i, idx in enumerate(indices):
        r, c = divmod(i, cols)
        img = image_bank[idx]
        resized = cv2.resize(img, (cell_size - 20, cell_size - 20))  # small margin between faces

        y_off = r * cell_size + 10
        x_off = c * cell_size + 10
        canvas[y_off:y_off + resized.shape[0], x_off:x_off + resized.shape[1]] = resized

        placed_labels.append(labels[idx])

    return canvas, placed_labels

In [ ]:
# ------------------------------------------------------------
# 7b: Generate a batch of composite test images and check
# how many faces get detected vs how many were actually placed
# ------------------------------------------------------------

def evaluate_group_detection(image_bank, labels, n_composites=20, n_people=4):
    rng = random.Random(123)
    detection_counts = []

    for i in range(n_composites):
        composite, placed_labels = make_composite(image_bank, labels, n_people=n_people, rng=rng)
        faces = app.get(composite)
        detected = len([f for f in faces if f.det_score >= 0.5])
        detection_counts.append((len(placed_labels), detected))

    total_placed = sum(p for p, d in detection_counts)
    total_detected = sum(d for p, d in detection_counts)

    print(f"Composites tested: {n_composites}, faces per composite: {n_people}")
    print(f"Total faces placed: {total_placed}, total faces detected: {total_detected}")
    print(f"Detection rate: {total_detected/total_placed:.3f}")

    # show worst cases (where detection undercounted the most)
    shortfalls = sorted(detection_counts, key=lambda x: x[1] - x[0])[:5]
    print(f"Worst 5 (placed, detected): {shortfalls}")

    return detection_counts


print("Testing detection on 4-person composites...")
evaluate_group_detection(image_bank, labels, n_composites=20, n_people=4)

print("\nTesting detection on 8-person composites (denser)...")
evaluate_group_detection(image_bank, labels, n_composites=20, n_people=8)



In [ ]:

# ------------------------------------------------------------
# 7c: End-to-end retrieval test on group photos.
# Build a small "event" of composite photos, then query with
# ONE person's solo reference photo and check if we correctly
# find every composite they appear in.
# ------------------------------------------------------------

def build_synthetic_event(image_bank, labels, n_photos=100, people_per_photo=(2, 6), rng=None):
    """
    Simulates an 'event': n_photos composite images, each with a random
    number of random people. Returns composites + ground truth (which
    identity appears in which photo index).
    """
    if rng is None:
        rng = random.Random(42)

    event_photos = []
    appearance_map = {}  # label -> set of photo indices they appear in

    for photo_idx in range(n_photos):
        n_people = rng.randint(*people_per_photo)
        composite, placed_labels = make_composite(image_bank, labels, n_people=n_people, rng=rng)
        event_photos.append(composite)

        for label in placed_labels:
            appearance_map.setdefault(label, set()).add(photo_idx)

    return event_photos, appearance_map


def index_event_photos(event_photos):
    """Detect + embed every face across the synthetic event."""
    embeddings = []
    meta = []
    for photo_idx, img in enumerate(event_photos):
        faces = app.get(img)
        for face in faces:
            if face.det_score < 0.5:
                continue
            embeddings.append(face.embedding)
            meta.append({"photo_idx": photo_idx})
    embeddings = np.array(embeddings)
    embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings_norm, meta


def query_event(query_img, embeddings_norm, meta, threshold=0.35):
    faces = app.get(query_img)
    if len(faces) == 0:
        return []
    query_emb = faces[0].embedding
    query_emb_norm = query_emb / np.linalg.norm(query_emb)
    sims = embeddings_norm @ query_emb_norm
    matches = np.where(sims >= threshold)[0]
    photo_indices = set(meta[i]["photo_idx"] for i in matches)
    return photo_indices


print("\nBuilding synthetic 100-photo event...")
event_photos, appearance_map = build_synthetic_event(image_bank, labels, n_photos=100)
event_embeddings, event_meta = index_event_photos(event_photos)
print(f"Indexed {len(event_embeddings)} faces across {len(event_photos)} photos")

# Test retrieval for a handful of people who appear in multiple event photos
test_people = [label for label, photos in appearance_map.items() if len(photos) >= 3][:5]

print(f"\nTesting retrieval for {len(test_people)} people who appear in 3+ photos:")
for person_label in test_people:
    # use their original LFW solo photo as the query
    solo_idx = np.where(labels == person_label)[0][0]
    query_img = image_bank[solo_idx]

    ground_truth = appearance_map[person_label]
    predicted = query_event(query_img, event_embeddings, event_meta, threshold=0.35)

    tp = len(predicted & ground_truth)
    fp = len(predicted - ground_truth)
    fn = len(ground_truth - predicted)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"  person {person_label} ({names[person_label]}): "
          f"appears in {len(ground_truth)} photos, found {len(predicted)}, "
          f"precision={precision:.2f} recall={recall:.2f}")

In [ ]:
# ============================================================
# Step 8: Show the actual matched images, not just counts
# Run this AFTER Step 7 (needs event_photos, event_embeddings, event_meta)
# ============================================================

import matplotlib.pyplot as plt

def query_event_with_images(query_img, embeddings_norm, meta, event_photos, threshold=0.35):
    """
    Same matching logic as query_event(), but returns the actual
    image arrays (not just indices) for display.
    """
    faces = app.get(query_img)
    if len(faces) == 0:
        print("No face detected in query image")
        return []

    query_emb = faces[0].embedding
    query_emb_norm = query_emb / np.linalg.norm(query_emb)
    sims = embeddings_norm @ query_emb_norm

    matches = np.where(sims >= threshold)[0]

    # dedupe by photo_idx, keep highest similarity score per photo
    photo_scores = {}
    for i in matches:
        photo_idx = meta[i]["photo_idx"]
        score = sims[i]
        if photo_idx not in photo_scores or score > photo_scores[photo_idx]:
            photo_scores[photo_idx] = score

    # sort by similarity, highest confidence first
    sorted_matches = sorted(photo_scores.items(), key=lambda x: -x[1])

    results = [(event_photos[photo_idx], photo_idx, score) for photo_idx, score in sorted_matches]
    return results


def show_matches(query_img, results, max_display=12):
    """
    Plots the query photo, then a grid of all matched photos with their
    similarity scores.
    """
    n_results = min(len(results), max_display)
    cols = 4
    rows = int(np.ceil((n_results + 1) / cols))  # +1 for query image

    plt.figure(figsize=(4 * cols, 4 * rows))

    # query image first
    plt.subplot(rows, cols, 1)
    plt.imshow(cv2.cvtColor(query_img, cv2.COLOR_BGR2RGB))
    plt.title("QUERY", fontsize=12, color="red", fontweight="bold")
    plt.axis("off")

    # matched images
    for i, (img, photo_idx, score) in enumerate(results[:max_display]):
        plt.subplot(rows, cols, i + 2)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(f"photo #{photo_idx}\nsim={score:.3f}", fontsize=10)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

    if len(results) > max_display:
        print(f"...and {len(results) - max_display} more matches not shown")



In [ ]:

# ------------------------------------------------------------
# Example usage: pick one of the test people and actually see
# the photos they were found in
# ------------------------------------------------------------

person_label = test_people[1]  # change index to try different people
solo_idx = np.where(labels == person_label)[0][0]
query_img = image_bank[solo_idx]

print(f"Querying for: {names[person_label]}")
results = query_event_with_images(query_img, event_embeddings, event_meta, event_photos, threshold=0.35)
print(f"Found in {len(results)} photos\n")

show_matches(query_img, results)